<a href="https://colab.research.google.com/github/pdf1802/f1-data-science/blob/main/notebooks/qualy_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏁 Tool 4 — Qualy-to-Race Predictor
**Question:** Is pole position real race pace?

Qualifying is a single-lap sprint on fresh Softs with minimal fuel.
The race is 50+ laps on Mediums/Hards with a full tank.

This tool applies three physics-based corrections to Q3 lap times
to predict race pace ranking — and highlights where qualifying rank
diverges from predicted race pace.

**Models used:** Module 1 (lap_time_model.pkl) only  
**New model trained:** None

> **V3.1 (Nov 2026) — one implementation, explained here.** The code of this tool now lives in
> [`tools/qualy_predictor.py`](../tools/qualy_predictor.py), shared by the dashboard, the Race Engineer and the tests. This notebook walks
> through it step by step: each step shows the **real source** (`show(...)`) and runs it on real races
> from `data/laps_2022_2025.parquet` with the V3.1 models. The earlier self-contained version (V2 models,
> its own copy of the code) is in the git history. Qualifying times come from FastF1, **offline from your cache** (the archive answers 401 since Oct 2026).

In [1]:
# Setup — local checkout or Colab. Data and models are not in git (see the README: Data, Train the models).
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
if "google.colab" in sys.modules:                       # Colab: clone the repo, models + Parquet from Drive
    if not os.path.exists("/content/f1-data-science"):
        subprocess.run(["git", "clone", "-q", "https://github.com/pdf1802/f1-data-science", "/content/f1-data-science"], check=True)
    os.chdir("/content/f1-data-science")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("F1_MODEL_DIR", "/content/drive/MyDrive/f1_models")
    os.environ.setdefault("F1_LAPS", "/content/drive/MyDrive/f1_data/laps_2022_2025.parquet")
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Code, display
from tools import shared, training

def show(obj):
    """Display the real source of a function / class from tools/ — the code the dashboard runs."""
    return Code(inspect.getsource(obj), language="python")

LAPS_ALL = pd.read_parquet(os.environ.get("F1_LAPS", os.path.join(ROOT, "data", "laps_2022_2025.parquet")))
LAPS, WET = training.dry_races(LAPS_ALL)
print(f"Models: {os.path.relpath(shared.resolve_model_dir(), ROOT)}/")
print(f"Laps: {LAPS_ALL.groupby(['Season', 'Race']).ngroups} races, {len(LAPS_ALL):,} laps "
      f"· {LAPS.groupby(['Season', 'Race']).ngroups} dry races used")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})

Models: models/
Laps: 43 races, 44,526 laps · 36 dry races used


In [2]:
# FastF1 for qualifying sessions: offline, from the cache (cache_drive/f1_cache or $F1_CACHE_DIR).
# F1 live timing answers 401 to downloads since Oct 2026, so only cached sessions load.
import fastf1
fastf1.set_log_level("ERROR")
CACHE = shared.enable_fastf1_cache()
print(f"FastF1 offline · cache with {len([p for p in os.scandir(CACHE) if p.is_dir()])} seasons")

FastF1 offline · cache with 4 seasons


## Data loading

`load_qualifying_results` takes each driver's **best lap across Q1–Q3** (V2 used `pick_quicklaps()` on Q3 only,
which left out everyone knocked out in Q1/Q2). Team names go through `shared.encode_team`, whose fuzzy fallback
handles names that drift across seasons ("Red Bull Racing" vs "Red Bull").

In [3]:
from tools import qualy_predictor as qp
show(qp.load_qualifying_results)

def load_qualifying_results(year: int, gp: str, session=None) -> pd.DataFrame:
    """Best lap per driver across Q1–Q3 → Driver | Team | QualyTime | QualyRank."""
    if session is None:
        import fastf1
        enable_fastf1_cache()
        session = fastf1.get_session(year, gp, "Q")
        session.load(laps=True, telemetry=False, weather=False, messages=False)

    laps = session.laps
    laps = laps[laps["LapTime"].notna()]
    idx = laps.groupby("Driver")["LapTime"].idxmin()
    best = laps.loc[idx, ["Driver", "Team", "LapTime"]].copy()
    best["QualyTime"] = best["LapTime"].dt.total_seconds()
    best = best.drop(columns="LapTime").sort_values("QualyTime").reset_index(drop=True)
    best["QualyRank"] = best.index + 1
    best["GapToPole"] = best["QualyTime"] - best["QualyTime"].iloc[0]
    best.attrs["event"] = f"{session.event['EventName']} {year}"
    best.attrs["event_name"] = session.event["EventName"]
    return best

## The Three Corrections

Each correction isolates one physical gap between qualifying and race conditions.

**Correction 1 — Fuel load (+3.5s flat)**  
Qualifying: ~8kg fuel. Race start: ~105kg. Rule of thumb: 0.03s/kg → ~2.9s.
We use 3.5s (conservative) to also absorb aero trim differences.
Applied identically to every driver — it's a car physics effect, not driver-specific.

**Correction 2 — Compound switch (Soft lap 1 → Medium/Hard lap 1)**  
We call Module 1 twice per driver: once with SOFT tyre_age=1, once with the
race start compound tyre_age=1. The delta is per-team because TeamEncoded
affects the lap time prediction — a Ferrari on Mediums degrades differently
than a Haas on Mediums.

**Correction 3 — Degradation average (mean LapDelta over STINT_LAPS laps)**  
Module 1 predicts LapDelta at each tyre age. We average across laps 1–20.
This captures the *cost of running tyres* over the opening stint —
the real reason some qualifying pace doesn't translate to race pace.

> **V3.1 — what happened to corrections 2 and 3.** The V3.1 lap model has **no team feature** any more: team turned
> out to be a race fingerprint that let the trees memorise races (Module 1, V3 → V3.1). So the compound switch and
> the degradation average are now the **same for every driver**. Together with the flat fuel term, all three
> corrections add the same number of seconds to everyone — and a constant can't reorder anything.
> We keep the tool and show exactly that below: it is a lesson in what a correction needs in order to matter.

In [4]:
show(qp.predict_race_pace)

def predict_race_pace(quali: pd.DataFrame, start_compound: str = "MEDIUM",
                      track_temp: float = DEFAULT_TRACK_TEMP,
                      air_temp: float = DEFAULT_AIR_TEMP,
                      stint_laps: int = STINT_LAPS, circuit: str | None = None) -> pd.DataFrame:
    """
    Apply the three corrections to a qualifying table (from load_qualifying_results).
    circuit: event name; defaults to the event stored on the table by load_qualifying_results.
    """
    df = quali.copy()
    df["TeamEncoded"] = df["Team"].map(encode_team)
    teams = df["TeamEncoded"].unique()
    race_c, soft_c = encode_compound(start_compound), encode_compound("SOFT")

    # One frame with every row we need, for every team → one predict() call
    circuit = circuit or quali.attrs.get("event_name")
    common = {"TrackTemp": track_temp, "AirTemp": air_temp, "RainfallEncoded": 0,
              "CircuitEncoded": encode_circuit(circuit)}
    comp_rows = pd.DataFrame([
        {"TeamEncoded": t, "CompoundEncoded": c, "TyreLife": 1, "LapNumber": CORRECTION_LAP,
         "Kind": kind, **common}
        for t in teams for c, kind in ((soft_c, "soft"), (race_c, "race"))])
    deg_rows = pd.DataFrame([
        {"TeamEncoded": t, "CompoundEncoded": race_c, "TyreLife": age, "LapNumber": age,
         "Kind": "deg", **common}
        for t in teams for age in range(1, stint_laps + 1)])
    rows = pd.concat([comp_rows, deg_rows], ignore_index=True)
    rows["Delta"] = predict_lap_deltas(rows)

    piv = rows[rows.Kind != "deg"].pivot_table(index="TeamEncoded", columns="Kind", values="Delta")
    compound_delta = (piv["race"] - piv["soft"]).clip(lower=0.0)   # a harder tyre is never faster
    deg_avg = rows[rows.Kind == "deg"].groupby("TeamEncoded")["Delta"].mean()

    df["FuelAdjTime"] = df["QualyTime"] + FUEL_CORRECTION
    df["CompoundDelta"] = df["TeamEncoded"].map(compound_delta)
    df["DegAverage"] = df["TeamEncoded"].map(deg_avg)
    df["PredictedRacePace"] = df["FuelAdjTime"] + df["CompoundDelta"] + df["DegAverage"]

    df = df.sort_values("PredictedRacePace").reset_index(drop=True)
    df["PredictedRaceRank"] = df.index + 1
    df["RankChange"] = df["QualyRank"] - df["PredictedRaceRank"]
    df["Direction"] = np.sign(df["RankChange"]).map({1: "↑", -1: "↓", 0: "→"})
    return df

## `predict_race_pace()` — The Orchestrator

This function is the public API of the whole tool.
It calls the three corrections in sequence and adds rank + direction columns.

**`RankChange` = QualyRank − PredictedRaceRank**  
Positive → driver moves UP from qualifying to predicted race pace (↑)  
Negative → driver moves DOWN (↓)  
Zero → no change (→)

A large positive RankChange for a mid-grid driver is interesting —
it means their car architecture suits race conditions more than qualifying.
Aston Martin in 2023 is the canonical example.

## Bahrain 2023

In [5]:
from tools.race_engine import actual_result
quali = qp.load_qualifying_results(2023, "Bahrain Grand Prix")
r = LAPS[(LAPS.Season == 2023) & (LAPS.Race == "Bahrain Grand Prix")]
pred = qp.predict_race_pace(quali, "SOFT", float(r.TrackTemp.mean()), float(r.AirTemp.mean()))
actual = actual_result(LAPS, 2023, "Bahrain Grand Prix")
pred["Actual"] = pred.Driver.map(actual)
print("Correction spread across drivers (s): compound",
      round(pred.CompoundDelta.max() - pred.CompoundDelta.min(), 4), "· degradation",
      round(pred.DegAverage.max() - pred.DegAverage.min(), 4))
pred[["Driver", "Team", "QualyTime", "QualyRank", "CompoundDelta", "DegAverage", "PredictedRacePace",
      "PredictedRaceRank", "Direction", "Actual"]].round(3).head(12)

Correction spread across drivers (s): compound 0.0 · degradation 0.0


,Driver,Team,QualyTime,QualyRank,CompoundDelta,DegAverage,PredictedRacePace,PredictedRaceRank,Direction,Actual
0,VER,Red Bull Racing,89.708,1,0.0,1.494,94.702,1,→,1.0
1,PER,Red Bull Racing,89.846,2,0.0,1.494,94.840,2,→,2.0
2,LEC,Ferrari,90.000,3,0.0,1.494,94.994,3,→,NaN
3,SAI,Ferrari,90.154,4,0.0,1.494,95.148,4,→,4.0
4,ALO,Aston Martin,90.336,5,0.0,1.494,95.330,5,→,3.0
5,RUS,Mercedes,90.340,6,0.0,1.494,95.334,6,→,7.0
6,HAM,Mercedes,90.384,7,0.0,1.494,95.378,7,→,5.0
7,HUL,Haas F1 Team,90.809,8,0.0,1.494,95.803,8,→,15.0
8,STR,Aston Martin,90.836,9,0.0,1.494,95.830,9,→,6.0
9,OCO,Alpine,90.914,10,0.0,1.494,95.908,10,→,NaN


## Validation — every race with a cached qualifying session

Qualifying order is the baseline any race-pace predictor has to beat. For every dry race whose qualifying session
is in the cache we compute the Spearman rank correlation with the real finishing order (`compare_with_actual`;
drivers who did not finish are left out — a DNF says nothing about pace).

In [6]:
import fastf1
rows = []
for (season, race), rr in LAPS.groupby(["Season", "Race"]):
    try:
        q = qp.load_qualifying_results(int(season), race)
    except Exception as e:                       # not in the cache
        continue
    p = qp.predict_race_pace(q, "MEDIUM", float(rr.TrackTemp.mean()), float(rr.AirTemp.mean()))
    c = qp.compare_with_actual(p, actual_result(LAPS, int(season), race))
    rows.append({"race": f"{season} {race.replace(' Grand Prix', '')}", "drivers": c["n"],
                 "rho_qualy": c["spearman_qualy"], "rho_model": c["spearman_model"]})
val = pd.DataFrame(rows).set_index("race").round(3)
print(f"{len(val)} races · mean ρ qualifying order {val.rho_qualy.mean():.3f} · model {val.rho_model.mean():.3f}")
val.sort_values("rho_qualy")

20 races · mean ρ qualifying order 0.862 · model 0.862


,drivers,rho_qualy,rho_model
race,,,
2024 Abu Dhabi,17,0.608,0.608
2023 Bahrain,17,0.730,0.730
2025 Abu Dhabi,20,0.767,0.767
2025 Bahrain,19,0.781,0.781
2025 Austrian,16,0.794,0.794
2024 Miami,19,0.818,0.818
2024 Australian,17,0.865,0.865
2025 Monaco,18,0.874,0.874
2024 Belgian,19,0.881,0.881


**What the numbers say.**

- Every correction is **identical across the grid** (compound switch 0 s from a SOFT start, degradation 1.49 s for
  everyone in Bahrain 2023), so the predicted order *is* the qualifying order: the same ρ (mean **0.862**) in all
  20 races. The tool adds nothing over the baseline — and the table proves it rather than hiding it.
- Qualifying order alone is a strong baseline: from ρ 0.61 (Abu Dhabi 2024) to 0.98 (Monaco 2024, where you can't
  overtake).
- Bahrain 2023 shows what is missing: ALO P5 → P3 and STR P9 → P6 (Aston Martin's race pace), HUL P8 → P15. The V2
  version did reorder by team — ALO predicted P1 — but that team term came from a lap model that had trained on the
  same races it was predicting. A correction that only works because it has seen the answer is not a correction.

## Known limitations — and the next step

- **No team or driver race pace.** The fix is not to put the team back into the lap model (that is what made it
  memorise races) but a separate, out-of-sample **race-craft term**: how each team's race pace compared with its
  qualifying pace in the *previous* races of the season. That is a prior that can actually reorder the grid.
- **Uniform fuel correction and circuit-average temperatures.**
- **Qualifying ≠ starting grid**: grid penalties are not applied.